In [1]:
import Pkg; Pkg.activate(@__DIR__); Pkg.instantiate()

  Activating project at `~/.julia/dev/drag-formation`


In [2]:
using LinearAlgebra
using SparseArrays

In [3]:
### Constants
μ_mks = 398600.44 #Earth standard gravitational parameter (km^3/sec^2)
J2 = 0.00108262668 #Earth J2 gravity coefficient (dimensionless)
Re_mks = 6378.137 #Radius of the Earth (km)
Ωe_mks = 7.292115e-5 #Rotation rate of Earth (rad/sec)

Cd = 2.2 #Drag coefficient (unitless)
AoM_mks = (2+sqrt(2))*3e-4*1e-4/3.0; #Area-to-mass ratio (km^2/kg) assumes a 3U CubeSat with deployable panels and mass of 3 kg
AoM0_mks = 1e-4*1e-4/3.0; #Area-to-mass ratio (km^2/kg) assumes a 3U CubeSat with deployable panels and mass of 3 kg

In [4]:
#Orbital Parameters (roughly ISS)
a_mks = 6791.0 #semi-major axis (km)
inc = 54.74 #inclination (deg)
n_mks = sqrt(μ_mks/a_mks^3) #mean motion (rad/sec)
P_mks = 2*pi/n_mks; #orbital period (sec) 1.54 hour 5569.440609858254 sec


In [5]:
#Normalize units
a = 1.0 #distance unit is semi-major axis
P = 1 #time unit is orbital period
μ = (2*pi/P)^2 * a^3 #μ_mks*(P_mks^2/a_mks^3)
n = sqrt(μ/a^3)
Re = Re_mks/a_mks
Ωe = Ωe_mks*P_mks;

#a0 = a0_mks/a_mks
#H = H_mks/a_mks
#ρ0 = ρ0_mks*(a_mks^3)
AoM = AoM_mks/(a_mks^2)
AoM0 = AoM0_mks/(a_mks^2);


In [59]:
### Coordinate and unit transformations

function cartesian_to_spherical(x)
    r = sqrt(x[1:3]'*x[1:3]) #range
    ϕ = atan(x[2],x[1]) #azimuth
    θ = atan(x[3],sqrt(x[1:2]'*x[1:2])) #elevation measured up from equatorial plane
    
    return [r; ϕ; θ]
end

function spherical_to_cartesian(s)
    r = s[1] #range
    ϕ = s[2] #azimuth
    θ = s[3] #elevation measured up from equatorial plane
    
    x = r*cos(θ)*cos(ϕ)
    y = r*cos(θ)*sin(ϕ)
    z = r*sin(θ)
    
    return [x; y; z]
end

function rtn_frame(x0)
    r = x0[1:3]
    v = x0[4:6]
    n = cross(r,v)
    
    R = r/norm(r)
    N = n/norm(n)
    T = cross(N,R)
    
    return [R T N] #rotation matrix from RTN to Cartesian frame
end

function rtn_to_cartesian(x0,δr)
    r0 = x0[1:3]
    v0 = x0[4:6]
    
    R = rtn_frame(x0) #rotation from RTN to Cartesian frame
    r = R*δr + r0 #position in inertial frame
    
    return r
end

function cartesian_to_rtn(x0,r)
    r0 = x0[1:3]
    v0 = x0[4:6]
    
    R = rtn_frame(x0) #rotation from RTN to Cartesian frame
    δr = R'*(r - r0) #position in RTN frame
    
    return δr
end

function cartesian_to_aθΩ(x)
    r = x[1:3]
    v = x[4:6]
        
    E = 0.5*v'*v + potential(cartesian_to_spherical(x))
    
    a = -μ/(2*E)
    
    i_x = [1,0,0]
    i_y = [0,1,0]
    i_z = [0,0,1]
    
    h = cross(r,v)
    Ω = atan(h[2]/norm(h[1:2]), h[1]/norm(h[1:2]))
    
    n = cross(i_z,h)/norm(cross(i_z,h))
    if dot(r,i_z) > 0
        θ = acos(dot(n,(r/norm(r))))
    else
        θ = 2*pi- acos(dot(n,(r/norm(r))))
    end
    return [a; θ; Ω]
end

function cartesian_to_ΔaθΩ(x0,x1)
    r0 = x0[1:3]
    v0 = x0[4:6]
    
    r1 = x1[1:3]
    v1 = x1[4:6]
    
    E0 = 0.5*v0'*v0 + potential(cartesian_to_spherical(x0))
    E1 = 0.5*v1'*v1 + potential(cartesian_to_spherical(x1))
    
    a0 = -μ/(2*E0)
    a1 = -μ/(2*E1)
    Δa = a1-a0
    

    R = rtn_frame(x0) #rotation from RTN to Cartesian frame    
    rtn1 = R'*r1
    Δθ = atan(rtn1[2]/norm(rtn1[1:2]), rtn1[1]/norm(rtn1[1:2]))
    
    h0 = cross(r0,v0)
    Ω0 = atan(h0[2]/norm(h0[1:2]), h0[1]/norm(h0[1:2]))
    
    h1 = cross(r1,v1)
    Ω1 = atan(h1[2]/norm(h1[1:2]), h1[1]/norm(h0[1:2]))
    
    ΔΩ = Ω1-Ω0
    
    return [Δa; Δθ; ΔΩ]
end

function mks_to_orbital(x)
    #change state from MKS units to orbital units
    
    return [x[1:3]/a_mks; x[4:6]*(P_mks/a_mks)]
end

function orbital_to_mks(x)
    #change state from orbital units to MKS
    
    return [x[1:3]*a_mks; x[4:6]*(a_mks/P_mks)]
end

orbital_to_mks (generic function with 1 method)

In [7]:
function density_mks(alt)
    #This fit is good from 250-500 km altitude
    #Units are km, kg
    a = 106.2
    b = -0.03243
    c = 1.988
    d = -0.01672
    ρ = a*exp(b*alt) + c*exp(d*alt)
end

function atm_density(r)
    #ρ = ρ0*exp((a0-r)/H)
    ρ = (a_mks^3)*density_mks(r*a_mks - Re_mks)
end

atm_density (generic function with 1 method)

In [8]:
#Drag perturbation at initial altitude

#Fd_mks = 0.5*Cd*density_mks(a_mks-Re_mks)*(AoM_mks)*(n_mks*a_mks)^2
#ȧ_mks = 2*(a_mks^2/μ_mks)*a_mks*n_mks*Fd_mks

#a_mid = a-5.0/a_mks #evaluate drag at average altitude of maneuver
Fd = -0.5*Cd*atm_density(a)*(AoM)*(n*a)^2*0.965#-15/a_mks
ȧ = 2*(a^2/μ)*(a)*n*Fd

-0.00033402452458653083

In [9]:
#Continuous-Time Linear model
#state is [Δθ; Δa; ΔΩ] where these are differences between satellite 1 and satellite 2
#Can treat drag input as positive part going to sat 2 and negative part going to sat 1
# delta is always with respect to sat 1

k1 = (21.0/4.0)*J2*Re*Re*sqrt(μ/a^9)*cosd(inc) #dΩ̇/da
k2 = -(3.0/2)*sqrt(μ/a^5) #dθ̇/da (dn/da) #(21.0/4.0)*J2*Re*Re*sqrt(μ/a^9)*(1-1.5*(sind(inc))^2)#
k3 = ȧ #dȧ/d(Drag) - 3*sqrt(μ*a)*J2*Re*Re/a^3*sind(inc)^2 

A0 = [0 k2 0; 0 0 0; 0 k1 0]
# B0 = [0 0; -k3 k3; 0 0]

3×3 Matrix{Float64}:
 0.0  -471.239     0.0
 0.0     0.0       0.0
 0.0     0.909286  0.0

In [10]:
#Discrete-time model
num_sat = 2 # number of couple of satellites 

A_bar = zeros(3*num_sat,3*num_sat) 
B_bar = zeros(3*num_sat,num_sat+1) 

# B_bar[1:3,1:2]=B0
for k =0:num_sat-1
    A_bar[1+(3*k):3+(3*k),1+(3*k):3+(3*k)] = A0
    B_bar[3*k+2,1] = -k3
    B_bar[3*k+2,2+k] = k3
#     B_bar[1+(3*k):3+(3*k),k+1] = -k2
#     B_bar[1+(3*k):3+(3*k),k+1:k+2] = B0
end


h = 0.1 #time step
A = I + h*A_bar
B = h*B_bar + 0.5*h*h*A_bar*B_bar;
B_bar

9×4 Matrix{Float64}:
 0.0           0.0           0.0           0.0
 0.000334025  -0.000334025   0.0           0.0
 0.0           0.0           0.0           0.0
 0.0           0.0           0.0           0.0
 0.000334025   0.0          -0.000334025   0.0
 0.0           0.0           0.0           0.0
 0.0           0.0           0.0           0.0
 0.000334025   0.0           0.0          -0.000334025
 0.0           0.0           0.0           0.0

In [11]:
#Initial and terminal constraints
l_12 = 3.0 # number of revolutions
l_13 = 6.0
l_14 = 8.0
l_15 = 4.0
l_16 = 5.0
Δx0 = zeros(size(A,1))
Δθf = 2.0*pi*[l_12,l_13,l_14,l_15,l_16] #one full revolution (can be any multiple of 2π) ,l_14
Δaf = 0; #end at the same altitude

# Right Ascension Separation
k4 = k1/k2
ΔΩf = k4*(Δθf)

broadcast.(rad2deg,ΔΩf)

5-element Vector{Float64}:
 -2.0839297027213823
 -4.167859405442765
 -5.557145873923687
 -2.7785729369618437
 -3.4732161712023046

In [12]:
#Inequality constraints
u_min = 0
u_max = 1
Δa_min = -15/a_mks /num_sat
Δa_max = 15/a_mks /num_sat

0.0007362685907819173

In [13]:
#Discretized Optimal Control Problem Parameters
Nx = size(A,1)     # number of state
Nu = size(B,2)     # number of controls
Tfinal = 1000 # final time (number of orbits)
Nt = Int(Tfinal/h)+1    # number of time steps
Nz = Nt*(Nx+Nu) #number of decision variables
thist = Array(range(0,h*(Nt-1), step=h));


In [14]:
#Useful Matrices

# decision variables are stacked z = [u1; x2; u2; x3; u3; ...; xf]
U1 = zeros(Nt,Nz,num_sat) #Matrix that picks out all u of sat 1
U2 = zeros(Nt,Nz,num_sat) #Matrix that picks out all u of sat n
X1 = zeros(Nt,Nz,num_sat)#Matrix that picks out all x1 = Δθ
X2 = zeros(Nt,Nz,num_sat)#Matrix that picks out all x2 = Δa
X3 = zeros(Nt,Nz,num_sat)#Matrix that picks out all x3 = ΔΘ

for i = 1:num_sat
    U1[:,:,i] = kron(Diagonal(I,Nt), [I zeros(1,(Nu-1+Nx))])
    U2[:,:,i] = kron(Diagonal(I,Nt), [zeros(1,i) I zeros(1,(Nu-i)+Nx-1)])
    X1[:,:,i] = kron(Diagonal(I,Nt),[zeros(1,Nu) zeros(1,(3*(i-1))) I zeros(1,3*(num_sat)-1-3*(i-1))])
    X2[:,:,i] = kron(Diagonal(I,Nt),[zeros(1,Nu) zeros(1,1) zeros(1,(3*(i-1))) I zeros(1,3*(num_sat)-2-3*(i-1))])
    X3[:,:,i] = kron(Diagonal(I,Nt),[zeros(1,Nu) zeros(1,2) zeros(1,(3*(i-1))) I zeros(1,3*(num_sat)-3-3*(i-1))])
end

X1F = zeros(num_sat,(Nu+Nx)*Nt);  #Matrix that picks out x1f = Δθf
X2F = zeros(num_sat,(Nu+Nx)*Nt); #Matrix that picks out x2f = Δaf
# X3F = zeros(num_sat,(Nu+Nx)*Nt); #Matrix that picks out x3f = ΔΘf

for i = num_sat:-1:1
    X1F[num_sat-i+1,end-(i)*3+1] = 1.0
    X2F[num_sat-i+1,end-(i)*3+2] = 1.0
#     X3F[num_sat-i+1,end-(i)*3+3] = 1.0
end


#Dynamics equality constraints C*z = d
C = sparse([[B -I zeros(Nx,(Nt-1)*(Nu+Nx))]; zeros(Nx*(Nt-1),Nu) [kron(Diagonal(I,Nt-1), [A B]) zeros((Nt-1)*Nx,Nx)] + [zeros((Nt-1)*Nx,Nx) kron(Diagonal(I,Nt-1),[zeros(Nx,Nu) Diagonal(-I,Nx)])]])
d = [-A*Δx0; zeros(Nx*(Nt-1))];


In [15]:
#Convex Formulation
using Convex

z = Variable(Nz)

objective = 0#sum(abs(U1[:,:,1]*z))

for i = 1:num_sat  
    objective += sum(abs(U2[:,:,i]*z)) + sum(abs(X1[:,:,i]*z - Δθf[i]));#sum(abs(X3[:,:,i]*z - ΔΩf[i])) + #sum(abs(U1[:,:,i]*z))
end

problem = minimize(objective);
problem.constraints += [C*z == d] #dynamics constraints


for i = 1:num_sat
    problem.constraints += [X2F[i,:]'*z == Δaf] #final altitude constraint
#     problem.constraints += [X3F[i,:]'*z == ΔΩf] #final altitude constraint
    problem.constraints += [U1[:,:,i]*z >= u_min; U1[:,:,i]*z <= u_max] #drag constraints
    problem.constraints += [U2[:,:,i]*z >= u_min; U2[:,:,i]*z <= u_max] #drag constraints
    problem.constraints += [X2[:,:,i]*z >= Δa_min; X2[:,:,i]*z <= Δa_max]; #altitude constraints
end

In [16]:
#using ECOS
#solve!(problem, MOI.OptimizerWithAttributes(ECOS.Optimizer, "maxit" => 10000))

using MosekTools
solve!(problem, Mosek.Optimizer)

Problem
  Name                   :                 
  Objective sense        : min             
  Type                   : LO (linear optimization problem)
  Constraints            : 58543           
  Cones                  : 0               
  Scalar variables       : 28520           
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 18027
Eliminator terminated.
Eliminator started.
Freed constraints in eliminator : 6
Eliminator terminated.
Eliminator - tries                  : 2                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.01            
Lin. dep.  - number                 : 0               
Presolve terminated. Time: 0.06    
Problem
  Name                   :                 
  Objective 

In [17]:
u1hist = zeros(size(U1)[1],num_sat) # sat 1 control
u2hist = zeros(size(U2)[1],num_sat) # sat n control
θhist = zeros(size(X1)[1]+1,num_sat)
ahist = zeros(size(X2)[1]+1,num_sat)
Ωhist = zeros(size(X3)[1]+1,num_sat)
for i = 1:num_sat
    u1hist[:,i] = U1[:,:,i]*z.value;
    u2hist[:,i] = U2[:,:,i]*z.value;
    θhist[:,i] = [Δx0[1]; X1[:,:,i]*z.value];
    ahist[:,i] = [Δx0[2]; X2[:,:,i]*z.value];
    Ωhist[:,i] = [Δx0[3]; X3[:,:,i]*z.value];
end


In [18]:
using Plots
plotlyjs()

mylabel = String[]
mylabel1 = String[]
mylabel2 = String[]
for i = 1:num_sat
    append!(mylabel,["Sat. 1-"*string(i+1)])
    append!(mylabel1,["Sat. 1"])
    append!(mylabel2,["Sat. "*string(i+1)])
end

# plot(thist,u1hist,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot(thist,u2hist,label=permutedims(mylabel2))
plot!(xlabel="Time, s", ylabel = "Control")

WebIO._IJuliaInit()

In [19]:
plot(thist[1:end],broadcast.(rad2deg,θhist[2:end,:]),label=permutedims(mylabel))
plot!(xlabel="Time, s", ylabel = "True Anomaly Difference, deg")

In [20]:
plot(thist,a_mks*ahist[2:end,:], label=permutedims(mylabel))
plot!(xlabel="Time, s", ylabel = "Semi-major axis Difference, km")


In [21]:
plot(thist,broadcast.(rad2deg,Ωhist[2:end,:]),label=permutedims(mylabel))
plot!(xlabel="Time, s", ylabel = "RAAN Difference, deg")

In [22]:
plot(thist,a_mks*Ωhist[2:end,:],label=permutedims(mylabel))
plot!(xlabel="Time, s", ylabel = "Spherical Distance, km")

In [23]:
### Full Orbital Dynamics with J2 and Drag

function potential(s)
    #Input is position in spherical coordinates
    r = s[1] #range
    ϕ = s[2] #azimuth
    θ = s[3] #elevation measured up from equatorial plane
    
#     U = -(μ/r) #spherical
#     U = -μ/r*(1+J2*Re*Re/r^2*((1.5)*sin(θ)^2-0.5))# - (0.5*J2*μ*Re*Re/r^3)*(3*(sin(θ)^2) - 1.0) #with J2

    U = -(μ/r) + (0.5*J2*μ*Re*Re/r^3)*(3*(cos(ϕ)^2) - 1.0) #with J2
end

function satellite_dynamics(x,u)
    #Newton's 2nd law
    
    r = x[1:3]
    v = x[4:6]
    r_noJ2 = x[7:9]
    v_noJ2 = x[10:12]

    pos_x = r[1]
    pos_y = r[2]
    pos_z = r[3]
    r_norm = sqrt(r[1]^2+r[2]^2+r[3]^2)
    r_norm_noJ2 = sqrt(r_noJ2[1]^2+r_noJ2[2]^2+r_noJ2[3]^2)
    
    g = +1.5*J2*μ*Re^2/r_norm^4*[
        pos_x/r_norm*(5*pos_z^2/r_norm^2-1);
        pos_y/r_norm*(5*pos_z^2/r_norm^2-1);
        pos_z/r_norm*(5*pos_z^2/r_norm^2-3)] - μ/r_norm^3*r
    
    
    v_rel = v #- cross([0;0;Ωe],r) #Assume atmosphere co-rotates with the Earth
    ρ = atm_density(sqrt(r[1]^2+r[2]^2+r[3]^2))
    Fdrag = -0.5*(u*AoM)*Cd*ρ*(sqrt(v_rel[1]^2+v_rel[2]^2+v_rel[3]^2))*v_rel

    v_rel_noJ2 = v_noJ2 #- cross([0;0;Ωe],r) #Assume atmosphere co-rotates with the Earth
    ρ_noJ2 = atm_density(sqrt(r_noJ2[1]^2+r_noJ2[2]^2+r_noJ2[3]^2))
    Fdrag_noJ2 = -0.5*(u*AoM)*Cd*ρ_noJ2*(sqrt(v_rel_noJ2[1]^2+v_rel_noJ2[2]^2+v_rel_noJ2[3]^2))*v_rel_noJ2

    v̇ = g + Fdrag
    v̇_noJ2 = -μ/(r_norm_noJ2)^3 * r_noJ2 +  Fdrag_noJ2
    return [v; v̇; v_noJ2; v̇_noJ2]
end


satellite_dynamics (generic function with 1 method)

In [24]:
#Initial conditions in Cartesian Coordinates
r0 = [a; 0; 0]
v0 = [0; cosd(inc)*sqrt(μ/a); sind(inc)*sqrt(μ/a)]
x0 = [r0; v0; r0; v0];

r1 = [a; 0; 0]
v1 = [0; cosd(inc)*sqrt(μ/(a)); sind(inc)*sqrt(μ/(a))]
x1 = [r1; v1; r1; v1];

tspan = (0.0, Tfinal+50)
Nt = Tfinal+150

200

In [25]:
#Interpolation of control input trajectory from solver

using Interpolations

function formation_dynamics!(ẋ,x,p,t)    
    x1 = x[1:12]
    x2 = x[13:24]

#     couple_n = p
#     uinterp1 = LinearInterpolation(thist, u1hist[:,couple_n])
#     uinterp2= LinearInterpolation(thist, u2hist[:,couple_n])
    if t < Tfinal
        u1interp = p[1]
        u2interp = p[2]
        u1 = u1interp(t)
        u2 = u2interp(t)
    else
        u1 = 0; u2 = 0;
    end
        
    ẋ1 = satellite_dynamics(x1,u1)
    ẋ2 = satellite_dynamics(x2,u2)
    
    ẋ .= [ẋ1;ẋ2]
end

formation_dynamics! (generic function with 1 method)

In [83]:
#Full Orbit Mechanics Simulation and save results
using DifferentialEquations
# Nt = 150
# Nt = Nt*3
temp = LinRange(tspan[1],tspan[2],Nt)
x0_soln = zeros(6*num_sat,Nt)
x1_soln = zeros(6*num_sat,Nt)

x0_noJ2_soln = zeros(6*num_sat,Nt)
x1_noJ2_soln = zeros(6*num_sat,Nt)

Δa_soln = zeros(num_sat,Nt)
Δθ_soln = zeros(num_sat,Nt)
ΔΩ_soln = zeros(num_sat,Nt)

r_soln1 = zeros(num_sat,Nt)
ϕ_soln1 = zeros(num_sat,Nt)
θ_soln1 = zeros(num_sat,Nt)
u_soln1 = zeros(num_sat,Nt)

r_soln2 = zeros(num_sat,Nt)
ϕ_soln2 = zeros(num_sat,Nt)
θ_soln2 = zeros(num_sat,Nt)
u_soln2 = zeros(num_sat,Nt)

a_soln1 = zeros(num_sat,Nt)
ν_soln1 = zeros(num_sat,Nt)
Ω_soln1 = zeros(num_sat,Nt)

a_soln2 = zeros(num_sat,Nt)
ν_soln2 = zeros(num_sat,Nt)
Ω_soln2 = zeros(num_sat,Nt)

r_soln1_noJ2 = zeros(num_sat,Nt)
ϕ_soln1_noJ2 = zeros(num_sat,Nt)
θ_soln1_noJ2 = zeros(num_sat,Nt)
u_soln1_noJ2 = zeros(num_sat,Nt)

r_soln2_noJ2 = zeros(num_sat,Nt)
ϕ_soln2_noJ2 = zeros(num_sat,Nt)
θ_soln2_noJ2 = zeros(num_sat,Nt)
u_soln2_noJ2 = zeros(num_sat,Nt)

for i = 1:num_sat
#     parameters = i 
    u1interp = LinearInterpolation(thist, u1hist[:,i])
    u2interp = LinearInterpolation(thist, u2hist[:,i])
    
    parameters = [u1interp, u2interp]
    ref_prob = ODEProblem(formation_dynamics!,[x0; x1],tspan,parameters)
    ref_soln = solve(ref_prob,Tsit5(); abstol=1e-7,reltol=1e-7);
    

    temp = LinRange(tspan[1],tspan[2],Nt)
    for k = 1:Nt
        x0_soln[6*(i-1)+1:6*(i-1)+6,k] = ref_soln(temp[k])[1:6]
        x0_noJ2_soln[6*(i-1)+1:6*(i-1)+6,k] = ref_soln(temp[k])[7:12]
        
        x1_soln[6*(i-1)+1:6*(i-1)+6,k] = ref_soln(temp[k])[13:18]
        x1_noJ2_soln[6*(i-1)+1:6*(i-1)+6,k] = ref_soln(temp[k])[19:24]
    
        rϕθ1_noJ2 = cartesian_to_spherical(x0_noJ2_soln[6*(i-1)+1:6*(i-1)+6,k])
        rϕθ2_noJ2 = cartesian_to_spherical(x1_noJ2_soln[6*(i-1)+1:6*(i-1)+6,k])
        rϕθ1 = cartesian_to_spherical(x0_soln[6*(i-1)+1:6*(i-1)+6,k])
        rϕθ2 = cartesian_to_spherical(x1_soln[6*(i-1)+1:6*(i-1)+6,k])
        ΔaθΩ = cartesian_to_ΔaθΩ(x0_soln[6*(i-1)+1:6*(i-1)+6,k], x1_soln[6*(i-1)+1:6*(i-1)+6,k])
        aθΩ1 = cartesian_to_aθΩ(x0_soln[6*(i-1)+1:6*(i-1)+6,k])
        aθΩ2 = cartesian_to_aθΩ(x1_soln[6*(i-1)+1:6*(i-1)+6,k])
    
    
        Δa_soln[i,k] = ΔaθΩ[1]
        Δθ_soln[i,k] = ΔaθΩ[2]
        ΔΩ_soln[i,k] = ΔaθΩ[3]

        a_soln1[i,k] = aθΩ1[1]
        ν_soln1[i,k] = aθΩ1[2]
        Ω_soln1[i,k] = aθΩ1[3]
        
        a_soln2[i,k] = aθΩ2[1]
        ν_soln2[i,k] = aθΩ2[2]
        Ω_soln2[i,k] = aθΩ2[3]
        
        r_soln1_noJ2[i,k] = rϕθ1_noJ2[1]
        ϕ_soln1_noJ2[i,k] = rϕθ1_noJ2[2]
        θ_soln1_noJ2[i,k] = rϕθ1_noJ2[3]
    
        r_soln2_noJ2[i,k] = rϕθ2_noJ2[1]
        ϕ_soln2_noJ2[i,k] = rϕθ2_noJ2[2]
        θ_soln2_noJ2[i,k] = rϕθ2_noJ2[3]
        
        r_soln1[i,k] = rϕθ1[1]
        ϕ_soln1[i,k] = rϕθ1[2]
        θ_soln1[i,k] = rϕθ1[3]
    
        r_soln2[i,k] = rϕθ2[1]
        ϕ_soln2[i,k] = rϕθ2[2]
        θ_soln2[i,k] = rϕθ2[3]
        if temp[k] < Tfinal
            u_soln1[i,k] = u1interp(temp[k])
            u_soln2[i,k] = u2interp(temp[k])
        else
        u_soln1[i,k] = 0; u_soln2[i,k] = 0;
        end
    end
end
tsamp = LinRange(tspan[1],tspan[2],Nt);

┌ Warning: Backwards compatability support of the new return codes to Symbols will be deprecated with the Julia v1.9 release. Please see https://docs.sciml.ai/SciMLBase/stable/interfaces/Solutions/#retcodes for more information
└ @ SciMLBase /home/josephine/.julia/packages/SciMLBase/QqtZA/src/retcodes.jl:355
┌ Warning: Backwards compatability support of the new return codes to Symbols will be deprecated with the Julia v1.9 release. Please see https://docs.sciml.ai/SciMLBase/stable/interfaces/Solutions/#retcodes for more information
└ @ SciMLBase /home/josephine/.julia/packages/SciMLBase/QqtZA/src/retcodes.jl:355
┌ Warning: Backwards compatability support of the new return codes to Symbols will be deprecated with the Julia v1.9 release. Please see https://docs.sciml.ai/SciMLBase/stable/interfaces/Solutions/#retcodes for more information
└ @ SciMLBase /home/josephine/.julia/packages/SciMLBase/QqtZA/src/retcodes.jl:355


In [27]:
# using Plots
# tsamp = LinRange(tspan[1],tspan[2],Nt)
plot(tsamp,x0_soln[1,:],color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp,x0_soln[2,:],color=:blue)
plot!(tsamp,x0_soln[3,:],color=:green)

plot(tsamp,x1_soln[1,:],color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp,x1_soln[2,:],color=:blue)
plot!(tsamp,x1_soln[3,:],color=:green)

# plot(tsamp,x0_noJ2_soln[1,:],color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
# plot!(tsamp,x0_noJ2_soln[2,:],color=:blue)
# plot!(tsamp,x0_noJ2_soln[3,:],color=:green)

# plot!(tsamp,x1_noJ2_soln[1,:],color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
# plot!(tsamp,x1_noJ2_soln[2,:],color=:blue)
# plot!(tsamp,x1_noJ2_soln[3,:],color=:green)


In [93]:
plot(tsamp,broadcast.(rad2deg,ν_soln1[1,:]),color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)


In [92]:
plot(tsamp,broadcast.(rad2deg,Ω_soln1[1,:]),color=:red)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp,broadcast.(rad2deg,Ω_soln2[1,:]),color=:blue)#,label=permutedims(mylabel1),ls=:dot, linewidth=2)

# length(Ω_soln1[1,:][Ω_soln1[1,:] .>3.])

In [28]:
tsamp = LinRange(tspan[1],tspan[2],Nt)
plot(tsamp,a_mks*Δa_soln',label=permutedims(mylabel))
plot!(xlabel="Time, s", ylabel = "Altitude Difference, km")


In [29]:
plot(tsamp,broadcast.(rad2deg,Δθ_soln'),label=permutedims(mylabel))
plot!(xlabel="Time, s", ylabel = "True Anomaly Difference, deg")

In [30]:
plot(tsamp,broadcast.(rad2deg,ΔΩ_soln'),label=permutedims(mylabel))
plot!(xlabel="Time, s", ylabel = "RAAN Difference, deg", ylim = [-10,5])

In [31]:
plot(tsamp,a_mks*ΔΩ_soln',label=permutedims(mylabel), ylim = [-100*1e1,10*1e2])
plot!(xlabel="Time, s", ylabel = "Spherical Distance, km")

In [32]:
plot(tsamp,u_soln1',color=:blue,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp,u_soln2',label=permutedims(mylabel2))
plot!(xlabel="Time, s", ylabel = "Control")


In [33]:
plot(tsamp,a_mks*r_soln1_noJ2'.-Re_mks,color=:blue,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp,a_mks*r_soln2_noJ2'.-Re_mks,label=permutedims(mylabel2))
for i=1:num_sat
    plot!([tsamp[1], tsamp[end]],[a_mks*r_soln1_noJ2[1]-Re_mks-Δa_max*a_mks*i, a_mks*r_soln1_noJ2[1]-Re_mks[1]-Δa_max*a_mks*i],color=:red, linewidth=2,label="Limit")
end
plot!(xlabel= "Time, s",ylabel= "Altitude, km")


In [34]:
plot(tsamp,ϕ_soln1_noJ2',color=:red,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp,ϕ_soln2_noJ2',label=permutedims(mylabel2))
plot!(xlabel= "Time, s",ylabel= "Azimuth, rad")

In [35]:
plot(tsamp,θ_soln1_noJ2',color=:red,label=permutedims(mylabel1),ls=:dot, linewidth=2)
plot!(tsamp,θ_soln2_noJ2',label=permutedims(mylabel2))
plot!(xlabel= "Time, s",ylabel= "Elevation, rad")

In [36]:
# Final RAAN
broadcast.(rad2deg,ΔΩ_soln[:,end]')

println("Targeted ΔΩ ",  broadcast.(rad2deg, ΔΩf[1:num_sat]), " deg" )
println("Reached ΔΩ ",  broadcast.(rad2deg, ΔΩ_soln[:,end]), " deg" )
println("Final Error ΔΩ ", broadcast.(rad2deg, (ΔΩf[1:num_sat] .- ΔΩ_soln[:,end])), " deg" )
println("Final Azimut Sat 1 ", broadcast.(rad2deg, ϕ_soln1[:,end]), " deg")
println("Final Azimut Sat Coupled ", broadcast.(rad2deg, ϕ_soln2[:,end]), " deg")

println("Final Elevation Sat 1 ", broadcast.(rad2deg, θ_soln1[:,end]), " deg")
println("Final Elevation Sat Coupled ", broadcast.(rad2deg, θ_soln2[:,end]), " deg")

Targeted ΔΩ [-2.0839297027213823, -4.167859405442765, -5.557145873923687] deg
Reached ΔΩ [-2.3579821954679505, -5.287528474792208, -7.456492413083359] deg
Final Error ΔΩ [0.274052492746568, 1.1196690693494435, 1.899346539159672] deg
Final Azimut Sat 1 [5.332071227735282, -5.910209286670962, -3.07021383272717] deg
Final Azimut Sat Coupled [131.9650058343369, -151.94944685353133, -114.97555186474739] deg
Final Elevation Sat 1 [47.453993416525854, 51.20682490823291, 50.4127702994933] deg
Final Elevation Sat Coupled [-54.154061499378656, -28.367664421940283, 22.11207444761306] deg


In [37]:
using MeshCat, GeometryBasics, CoordinateTransformations, Rotations
using Plots

green_material = MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(0.5, 0.9, 0.5, 0.8))
red_material = MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(0.9, 0.5, 0.5, 0.8))
blue_material = MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(0.5, 0.5, 0.9, 0.3))


vis = Visualizer()
open(vis)
IJuliaCell(vis)
setprop!(vis["/Background"], "top_color", colorant"transparent") # makes it cool night style

# create earth
origin = [0, 0, 0] # Might be different if from COM or from edge
len = 2.0
radius = Re
earth = Sphere(Point(0.0, 0.0, 0.0), radius)


# Add Object to visualizer
setobject!(vis["earth"], earth, blue_material) 



size_spacecraft = Vec(0.05,0.05,0.05)
size_plate = Vec(size_spacecraft[1]*3,0.01,0.04)

origin_spacecraft = Vec(x0_soln[1,1]-size_spacecraft[1]/2,x0_soln[2,1]-size_spacecraft[2]/2,x0_soln[3,1]-size_spacecraft[3]/2)
origin_plate = Vec(x0_soln[1,1]-size_plate[1]/2,x0_soln[2,1]-size_plate[2]/2,x0_soln[3,1]-size_plate[3]/2)


cubeshape = HyperRectangle(origin_spacecraft,size_spacecraft)
plateshape =HyperRectangle(origin_plate,size_plate)
center = Sphere(Point(origin_spacecraft[1],origin_spacecraft[2],origin_spacecraft[3]), 0.01)

colors = []
for i =1: num_sat+1
    append!(colors,rand(Float64, (3)))
    spacecraft_material=MeshPhongMaterial(wireframe=false, wireframeLinewidth=0.001, side=2, color=RGBA(colors[(i-1)*3+1], colors[(i-1)*3+2], colors[(i-1)*3+3], 0.5))
    name_sat = "satellite"*string(i)
    name_plate = "plate"*string(i)
    setobject!(vis[name_sat], cubeshape, spacecraft_material) 
    setobject!(vis[name_sat][name_plate], plateshape, spacecraft_material)
end

sleep(5)
for i = 1:length(x0_soln[1,:])-1
    for j = 1 : num_sat
        if j == 1
#             positions_1 = [x0_noJ2_soln[1+(6*j-6),i]-origin_spacecraft[1],x0_noJ2_soln[2+(6*j-6),i]-origin_spacecraft[2],x0_noJ2_soln[3+(6*j-6),i]-origin_spacecraft[3]]
            positions_1 = [x0_soln[1+(6*j-6),i]-origin_spacecraft[1],x0_soln[2+(6*j-6),i]-origin_spacecraft[2],x0_soln[3+(6*j-6),i]-origin_spacecraft[3]]
            rotations_1 = i*2*pi/length(x0_soln[1+(6*j-6),:])
            position_1 = Translation(positions_1)
#             rotation_1 = LinearMap(RotZ(rotations_1))
            name_sat = "satellite"*string(j)
            settransform!(vis[name_sat], position_1)
#             settransform!(vis[name_sat], compose(position_1, rotation_1))
        end 
#         positions_2 = [x1_noJ2_soln[1+(6*j-6),i]-origin_spacecraft[1],x1_noJ2_soln[2+(6*j-6),i]-origin_spacecraft[2],x1_noJ2_soln[3+(6*j-6),i]-origin_spacecraft[3]]
        positions_2 = [x1_soln[1+(6*j-6),i]-origin_spacecraft[1],x1_soln[2+(6*j-6),i]-origin_spacecraft[2],x1_soln[3+(6*j-6),i]-origin_spacecraft[3]]
#         rotations_2 = i*2*pi/length(x1_soln[1+(6*j-6),:])
        position_2 = Translation(positions_2)
#         rotation_2 = LinearMap(RotZ(rotations_2))
        name_sat = "satellite"*string(j+1)
        settransform!(vis[name_sat], position_2)
#         settransform!(vis[name_sat], compose(position_2, rotation_2))
    end
        #     settransform!(vis["plate1"], position)
    sleep(0.5)
end


┌ Info: MeshCat server started. You can open the visualizer by visiting the following URL in your browser:
│ http://127.0.0.1:8700
└ @ MeshCat /home/josephine/.julia/packages/MeshCat/Ax8pH/src/visualizer.jl:73


Stub sandbox ignoring command: /app/extra/nacl_helper
Opening in existing browser session.


Gtk-Message: 12:54:46.342: Failed to load module "appmenu-gtk-module"
Gtk-Message: 12:54:46.398: Failed to load module "canberra-gtk-module"
Gtk-Message: 12:54:46.401: Failed to load module "canberra-gtk-module"
[2 zypak-helper] Wait found events, but sd-event found none
[2:2:0222/125446.739163:ERROR:nacl_fork_delegate_linux.cc(313)] Bad NaCl helper startup ack (0 bytes)
Fontconfig error: Cannot load default config file: No such file: (null)
Gtk-Message: 12:54:46.828: Failed to load module "appmenu-gtk-module"
Gtk-Message: 12:54:46.866: Failed to load module "canberra-gtk-module"
Gtk-Message: 12:54:46.867: Failed to load module "canberra-gtk-module"
Fontconfig error: Cannot load default config file: No such file: (null)


In [38]:
# using NaNStatistics
# window_size = length(tsamp)/1.0
# z1 = movmean(θ_soln1, window_size)
# z2 = movmean(θ_soln2, window_size)
# plot(tsamp,θ_soln1,linestyle=:dot)
# plot!(tsamp,θ_soln2,linestyle=:dot)
# plot!(tsamp,z1, linewidth = 2)
# plot!(tsamp,z2, linewidth = 2)

In [39]:
# window_size = length(tsamp)/100.0
# z1 = movmean(Ω_soln1, window_size)
# z2 = movmean(Ω_soln2, window_size)
# # plot(tsamp,Ω_soln1,linestyle=:dot)
# # plot!(tsamp,Ω_soln2,linestyle=:dot)
# plot(tsamp,z1, linewidth = 2)
# plot!(tsamp,z2, linewidth = 2)
# plot!(ylim= [-1,1])

In [40]:
k1/k2

-0.0019295645395568358

In [41]:
(k1/k2)*2*pi

-0.012123811564198254

In [42]:
((7.0/2)*(Re*Re/(a*a))*J2*cosd(inc))*2*pi

0.012123811564198252